# 04 - Model comparison, calibration and decision analysis
Runs the same code path as `scripts/train.py` in lightweight mode and inspects its results. Artifacts go to a temporary directory so the repository stays clean.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from credit_risk.config import load_config
from credit_risk.data.loader import load_dataset, dataset_status_report
cfg = load_config()
print(dataset_status_report(cfg))
# demo=True is an EXPLICIT opt-in: if the real Home Credit files are missing, SYNTHETIC data is generated.
df, DATA_SOURCE = load_dataset(cfg, demo=True)
print("\nDATA SOURCE:", DATA_SOURCE, df.shape)
if DATA_SOURCE != "home_credit":
    print("WARNING: SYNTHETIC data - numbers are only a code-path demonstration, not real results.")

In [ ]:
import tempfile
from credit_risk.config import apply_lightweight
from credit_risk.models.train import run_training
cfg_lw = apply_lightweight(cfg); cfg_lw['mlflow']['enabled'] = False
tmp = Path(tempfile.mkdtemp())
result = run_training(cfg_lw, df.sample(min(len(df), 30000), random_state=0), DATA_SOURCE, tmp/'artifacts', tmp/'reports', mlflow_enabled=False, run_shap=True)
print('selected:', result['selected_model'], '|', result['selection_rule'])

In [ ]:
pd.DataFrame(result['comparison'])[['model','val_roc_auc','test_roc_auc','test_pr_auc','test_ks','test_brier_uncalibrated','test_brier_calibrated','calibration','best_params']]

In [ ]:
from IPython.display import Image, display
for f in ['roc_pr_test.png','calibration_selected_model.png','threshold_sensitivity.png']: display(Image(filename=str(tmp/'reports'/'figures'/f)))

## Policy scenarios (illustrative thresholds and costs)
A higher flag threshold approves more applicants (fewer false positives) but misses more defaulters (more false negatives). The cost weights in `configs/config.yaml` are arbitrary and should be replaced with business estimates.

In [ ]:
pd.DataFrame(result['policy_comparison'])

In [ ]:
pd.DataFrame(result['tier_summary'])